# Charging Event Coverage

Load the flight parquet files and count how many charging events are available in the dataset.

In [1]:
from pathlib import Path
import importlib

if importlib.util.find_spec('pyarrow') is None and importlib.util.find_spec('fastparquet') is None:
    raise ImportError(
        "Parquet support is not installed in this kernel. Run the setup cell above, then restart the kernel and rerun the notebook."
    )

import pandas as pd
import pyarrow as pa

# These parquet files appear to contain legacy Python extension type metadata.
# Arrow <= 20 can still read them if PyExtensionType auto-loading is enabled.
if hasattr(pa, 'PyExtensionType'):
    pa.PyExtensionType.set_auto_load(True)

print('pandas', pd.__version__)
print('pyarrow', pa.__version__)

pandas 2.3.3
pyarrow 20.0.0


In [ ]:
def find_parquet(*relative_candidates: str) -> Path:
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        for relative_path in relative_candidates:
            candidate = (root / relative_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not find any of: {relative_candidates}")


manifest_path = find_parquet(
    'data/event_manifest.parquet',
    'data/processed/event_manifest.parquet',
)
timeseries_path = find_parquet(
    'data/event_timeseries.parquet',
    'data/processed/event_timeseries.parquet',
)

manifest_path, timeseries_path

(PosixPath('/Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/data/event_manifest.parquet'),
 PosixPath('/Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/data/event_timeseries.parquet'))

: 

In [ ]:
manifest = pd.read_parquet(manifest_path)
timeseries = pd.read_parquet(timeseries_path)

print(f'manifest shape: {manifest.shape}')
print(f'timeseries shape: {timeseries.shape}')

In [ ]:
manifest[['flight_id', 'plane_id', 'registration', 'event_datetime', 'event_type_main', 'is_charging_event']].head()

In [ ]:
charging_events = manifest.loc[manifest['is_charging_event'] == 1].copy()

summary = pd.DataFrame(
    {
        'metric': [
            'total manifest events',
            'charging events',
            'unique planes with charging events',
            'unique charging event flight_ids',
        ],
        'value': [
            len(manifest),
            len(charging_events),
            charging_events['plane_id'].nunique(),
            charging_events['flight_id'].nunique(),
        ],
    }
)

summary

In [ ]:
charging_events_by_plane = (
    charging_events.groupby(['plane_id', 'registration'], dropna=False)
    .agg(
        charging_event_count=('flight_id', 'nunique'),
        first_event=('event_datetime', 'min'),
        last_event=('event_datetime', 'max'),
    )
    .sort_values(['charging_event_count', 'plane_id'], ascending=[False, True])
    .reset_index()
)

charging_events_by_plane

In [ ]:
charging_sample_rows = timeseries.loc[timeseries['is_charging_event'] == 1].copy()

timeseries_summary = pd.DataFrame(
    {
        'metric': [
            'charging telemetry rows',
            'unique charging events in timeseries',
            'unique charging source csv files',
        ],
        'value': [
            len(charging_sample_rows),
            charging_sample_rows['flight_id'].nunique(),
            charging_sample_rows['source_csv_name'].nunique() if 'source_csv_name' in charging_sample_rows.columns else pd.NA,
        ],
    }
)

timeseries_summary

In [ ]:
charging_events[['flight_id', 'plane_id', 'registration', 'event_datetime', 'detail_flight_type', 'event_type_main']].sort_values(['plane_id', 'event_datetime']).head(20)